# Post-Processing Demo: Null Handling

When using constrained-output tools (Outlines, Instructor, JSON mode), the LLM
always produces all schema fields. It signals "I don't know" by outputting `null`
or `""`, not by omitting the key.

By default, the evaluator treats `null` as a value:
- `null` vs `"PVD"` = mismatch (wrong value)
- `null` vs `null` = match (same value)

But with `reclassify_nulls` as a post-processor, absent values are reclassified:
- `null` vs `"PVD"` = hallucination (gold has no answer, extractor invented one)
- `"PVD"` vs `null` = omission (gold has answer, extractor gave up)
- `null` vs `null` = skipped (both absent, excluded from metrics)

In [ ]:
from struct_extract_eval import evaluate, infer_schema, annotate_xeval
from struct_extract_eval.postprocess import NullHandling, reclassify_nulls
from copy import deepcopy

## Data

A materials science extraction scenario. The LLM uses constrained output so every
field is always present, but uses `null` or `""` to mean "I couldn't find this."

- Record 0: Gold has all fields. Extracted gives up on `temp` (null) and `notes` ("").
- Record 1: Gold has no `notes` (null). Extracted invents notes that don't exist in gold.

In [ ]:
GOLD = [
    {"method": "PVD", "temp": 300, "notes": "Substrate pre-heated."},
    {"method": "CVD", "temp": 500, "notes": None},
]

EXTRACTED = [
    {"method": "PVD", "temp": None, "notes": ""},    # gave up on temp and notes
    {"method": "CVD", "temp": 500, "notes": "N/A"},  # invented notes (gold is null)
]

## Schema

In [ ]:
schema = infer_schema(GOLD)
eval_schema = deepcopy(schema)
annotate_xeval(eval_schema)

import json
print(json.dumps(eval_schema, indent=2))

## Step 1: Default scoring (no post-processing)

`null` and `""` are treated as regular values. `null` vs `300` = mismatch.

In [ ]:
result_default = evaluate(GOLD, EXTRACTED, schema=eval_schema)

print("Default scoring (null is a value):")
print(f"  Mean F1: {result_default.mean_f1:.3f}")
print()
for record in result_default.records:
    print(f"  Record {record.record_id} -- F1: {record.f1:.3f}")
    for fr in record.field_results:
        print(f"    {fr.path:<15} {fr.score:.1f}  {fr.status:<12} "
              f"gold={str(fr.gold_value):<20} ext={str(fr.extracted_value)}")
    print()

Problems:
- `temp`: null vs 300 is a "mismatch" -- but the extractor didn't guess wrong,
  it **gave up**. Should be an omission.
- `notes` (record 0): "" vs "Substrate pre-heated." is a "mismatch" -- same issue.
- `notes` (record 1): null vs "N/A" is a "mismatch" -- but gold has no answer.
  The extractor **invented** something. Should be a hallucination.
- `notes` (record 1): if gold is null and extracted is also null, it's a "match" --
  but there's nothing meaningful to match on. Should be skipped.

## Step 2: With null handling post-processor

Configure: `None` and `""` both mean "absent." Both absent = skip.

In [ ]:
config = NullHandling(
    absent_values=[None, ""],  # both mean "I don't know"
    both_absent_skip=True,     # null vs null = skip (not match)
)

result_null = evaluate(
    GOLD, EXTRACTED, schema=eval_schema,
    post_process=lambda frs: reclassify_nulls(frs, config),
)

print("With null handling (null/empty = absent):")
print(f"  Mean F1: {result_null.mean_f1:.3f}")
print()
for record in result_null.records:
    print(f"  Record {record.record_id} -- F1: {record.f1:.3f}")
    for fr in record.field_results:
        reason = f"  ({fr.reason})" if fr.reason else ""
        print(f"    {fr.path:<15} {fr.score:.1f}  {fr.status:<15} "
              f"gold={str(fr.gold_value):<20} ext={str(fr.extracted_value)}{reason}")
    print()

Now the statuses correctly reflect what happened:

| Field | Gold | Extracted | Default | With null handling |
|-------|------|-----------|---------|--------------------|
| method | "PVD" | "PVD" | match | match (unchanged) |
| temp (rec 0) | 300 | null | mismatch | **omission** (extractor gave up) |
| notes (rec 0) | "Substrate..." | "" | mismatch | **omission** (extractor gave up) |
| notes (rec 1) | null | "N/A" | mismatch | **hallucination** (gold has no answer) |

Omissions hurt recall. Hallucinations hurt precision. Skipped fields are excluded entirely.

## Step 3: Side-by-side comparison

In [ ]:
print(f"{'':25} {'Default':>10} {'Null handling':>14}")
print(f"{'':25} {'─' * 10} {'─' * 14}")
print(f"{'Mean F1':25} {result_default.mean_f1:>10.3f} {result_null.mean_f1:>14.3f}")
print(f"{'Mean Precision':25} {result_default.mean_precision:>10.3f} {result_null.mean_precision:>14.3f}")
print(f"{'Mean Recall':25} {result_default.mean_recall:>10.3f} {result_null.mean_recall:>14.3f}")
print(f"{'Omissions':25} {result_default.total_omissions:>10} {result_null.total_omissions:>14}")
print(f"{'Hallucinations':25} {result_default.total_hallucinations:>10} {result_null.total_hallucinations:>14}")

## How `absent_values` works

All values in the list are treated as **equivalent** absent markers:

```python
absent_values=[None, ""]  # both mean "no answer"
```

This means `None` vs `""` = both absent (skipped or match), NOT a mismatch.
If you consider them semantically different, only list one:

```python
absent_values=[None]   # only null means absent; "" is a real value
```

## Writing your own post-processor

`post_process` accepts any function with signature
`(list[FieldResult]) -> list[FieldResult]`:

```python
def my_custom_handler(field_results):
    for fr in field_results:
        # your logic here
        pass
    return field_results

result = evaluate(gold, extracted, schema, post_process=my_custom_handler)
```

Chain multiple post-processors:

```python
def chain(frs):
    frs = reclassify_nulls(frs, config)
    frs = my_custom_filter(frs)
    return frs

result = evaluate(gold, extracted, schema, post_process=chain)
```